# Proyecto Wearable: Transformación y Vectorización (Machine Learning)
**Curso:** Minería de Datos  
**Metodología:** CRISP-DM (Fase de Preparación y Transformación de Datos)  
**Objetivo:** Preparar la matriz numérica final para la ingesta de algoritmos de Machine Learning mediante escalado selectivo y codificación categórica.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

def encontrar_raiz():
    candidatos = [Path.cwd(), Path.cwd().parent]
    for c in candidatos:
        if (c / 'Datacrude').is_dir():
            return c
    return Path.cwd().parent

raiz_proyecto = encontrar_raiz()
if str(raiz_proyecto) not in sys.path:
    sys.path.append(str(raiz_proyecto))

print('Entorno configurado correctamente. Raiz del proyecto:', raiz_proyecto)

Entorno configurado correctamente. Raiz del proyecto: c:\Users\Juan Sebastian\OneDrive\Documentos\MineriaDatosUP\Data-MinerProyecto


---
## 1. Carga del Dataset Intermedio Limpio
Se realiza la lectura del archivo resultante de la fase de limpieza (`Sensores_limpio.csv`) para verificar su estructura previa a la vectorización.

In [2]:
def encontrar_intermedio():
    candidatos = [
        raiz_proyecto / 'Datacrude' / '02IntermediateData' / 'Sensores_limpio.csv',
        Path('../Datacrude/02IntermediateData/Sensores_limpio.csv'),
        Path('Datacrude/02IntermediateData/Sensores_limpio.csv'),
    ]
    for c in candidatos:
        if c.exists():
            return c
    raise FileNotFoundError('No se encontro Sensores_limpio.csv')

ruta_intermedia = encontrar_intermedio()
df_limpio = pd.read_csv(ruta_intermedia)

print('=== DIAGNOSTICO DEL DATASET INTERMEDIO ===')
print(f'Dimensiones del dataset: {df_limpio.shape}')
display(df_limpio.head(3))

=== DIAGNOSTICO DEL DATASET INTERMEDIO ===
Dimensiones del dataset: (868, 18)


,Id,ActivityDate,TotalSteps,TotalDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,Nivel_Pasos,Nivel_Sedentarismo,TotalMinutesAsleep,TotalTimeInBed,Eficiencia_Sueño
0,1503960366,2016-12-04,13162.0,8.500000e+00,1.880000e+14,0.55,6.060000e+14,0.0,25.0,13,328,728,1985.0,Alta Actividad,Alto Sedentarismo,NaN,NaN,NaN
1,1503960366,2016-04-13,10735.0,6.970000e+14,1.570000e+14,0.69,4.710000e+14,0.0,21.0,19,217,776,1797.0,Alta Actividad,Alto Sedentarismo,384.0,407.0,0.94
2,1503960366,2016-04-14,10460.0,6.740000e+14,2.440000e+14,0.40,3.910000e+14,0.0,30.0,11,181,1218,1776.0,Alta Actividad,Alto Sedentarismo,NaN,NaN,NaN


---
## 2. Fase de Transformación Modular (Pipeline de Escalado y Codificación)
Aplicación de la clase orientada a objetos `SensoresDataTransformer` ubicada en `src/Transformacion/transformaciones.py`:
1. **RobustScaler** y **MinMaxScaler** para variables numéricas.
2. **One-Hot Encoding** (Dummies) para las variables categóricas ordenadas (`Nivel_Pasos`, `Nivel_Sedentarismo`).

In [3]:
from src.Transformacion.transformaciones import SensoresDataTransformer

transformador = SensoresDataTransformer()
df_transformado = transformador.ejecutar_transformacion()
transformador.exportar_matriz()

=== PASO 1: CARGA DE DATOS INTERMEDIOS ===
Dataset cargado desde: C:\Users\Juan Sebastian\OneDrive\Documentos\MineriaDatosUP\Data-MinerProyecto\Datacrude\02IntermediateData\Sensores_limpio.csv
Dimensiones iniciales: (868, 18)

=== PASO 2: ESCALADO NUMÉRICO SELECTIVO ===
RobustScaler aplicado a: ['TotalSteps', 'Calories', 'VeryActiveMinutes']
MinMaxScaler aplicado a: ['TotalMinutesAsleep', 'TotalTimeInBed', 'SedentaryMinutes', 'Eficiencia_Sueño']

=== PASO 3: CODIFICACIÓN CATEGÓRICA (ONE-HOT ENCODING) ===
Variables codificadas mediante One-Hot Encoding: ['Nivel_Pasos', 'Nivel_Sedentarismo']
Nuevas columnas generadas: ['Nivel_Pasos_Baja Actividad', 'Nivel_Sedentarismo_Bajo Sedentarismo']

=== PASO 4: EXPORTACIÓN DE LA MATRIZ PROCESADA ===
Matriz procesada exportada exitosamente a: C:\Users\Juan Sebastian\OneDrive\Documentos\MineriaDatosUP\Data-MinerProyecto\Datacrude\03DatafinalProcessed\Sensores_transformado.csv
Dimensiones finales para modelado: (868, 18)


---
## 3. Inspección y Validación de la Matriz Procesada Final
Se verifica las dimensiones definitivas, la presencia de variables codificadas en formato binario y la correcta escala numérica de los datos preparados para la minería.

In [4]:
print('=== MUESTRA DE LA MATRIZ TRANSFORMADA FINAL ===')
display(df_transformado.head(5))

print('\n=== INFORMACION DE TIPOS Y NULOS EN LA MATRIZ FINAL ===')
print(df_transformado.info())

=== MUESTRA DE LA MATRIZ TRANSFORMADA FINAL ===


,Id,ActivityDate,TotalSteps,TotalDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,TotalMinutesAsleep,TotalTimeInBed,Eficiencia_Sueño,Nivel_Pasos_Baja Actividad,Nivel_Sedentarismo_Bajo Sedentarismo
0,1503960366,2016-12-04,0.833514,8.500000e+00,1.880000e+14,5.500000e-01,6.060000e+14,0.0,0.514286,13,328,0.505556,-0.239570,NaN,NaN,NaN,0,0
1,1503960366,2016-04-13,0.443963,6.970000e+14,1.570000e+14,6.900000e-01,4.710000e+14,0.0,0.400000,19,217,0.538889,-0.432045,0.453911,0.381696,0.88,0,0
2,1503960366,2016-04-14,0.399823,6.740000e+14,2.440000e+14,4.000000e-01,3.910000e+14,0.0,0.657143,11,181,0.845833,-0.453545,NaN,NaN,NaN,0,0
3,1503960366,2016-04-15,0.287789,6.280000e+14,2.140000e+14,1.260000e+14,2.830000e+14,0.0,0.628571,34,209,0.504167,-0.485283,0.493017,0.420759,0.86,1,0
4,1503960366,2016-04-16,0.754384,8.160000e+14,2.710000e+14,4.100000e-01,5.040000e+14,0.0,0.828571,10,221,0.536806,-0.364474,0.392458,0.337054,0.86,0,0



=== INFORMACION DE TIPOS Y NULOS EN LA MATRIZ FINAL ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 868 entries, 0 to 867
Data columns (total 18 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Id                                    868 non-null    int64  
 1   ActivityDate                          868 non-null    object 
 2   TotalSteps                            868 non-null    float64
 3   TotalDistance                         868 non-null    float64
 4   VeryActiveDistance                    868 non-null    float64
 5   ModeratelyActiveDistance              868 non-null    float64
 6   LightActiveDistance                   868 non-null    float64
 7   SedentaryActiveDistance               868 non-null    float64
 8   VeryActiveMinutes                     868 non-null    float64
 9   FairlyActiveMinutes                   868 non-null    int64  
 10  LightlyActiveMinutes         